# División (Split) del dataset de perfiles completos

Con el fin de preparar los datos para el entrenamiento de modelos, en este notebook se genera una metadata persistente con una partición estratificada.

El objetivo es que otros notebooks puedan reutilizar exactamente la misma división `train` / `val` / `test` sin recalcularla.

El resultado se guarda como `metadata.csv` con una nueva columna `split`.

In [1]:
# Importaciones
import pandas as pd

from vasijas.config import load_config
from vasijas.metadata import cargar_metadata, guardar_metadata
from vasijas.paths import ProjectPaths
from vasijas.split_dataset import (
    ConfiguracionSplitDataset,
    preparar_metadata_split_perfiles_completos,
)

In [2]:
# Configuracion general
CONFIG = load_config()
PATHS = ProjectPaths.from_root(CONFIG.project_root)

## Origen y destino de datos

In [3]:
# Origen y destino de datos
DATA_ORIGEN = PATHS.DATA_CUADRADOS
DATA_DESTINO = PATHS.DATA_DIVIDIDO
print(f"Dataset de imágenes a dividir: {DATA_ORIGEN}")
print(f"Destino del nuevo archivo de metadata: {DATA_DESTINO}")

Dataset de imágenes a dividir: /home/rafael/Dev/vasijas/data/05.cuadrados
Destino del nuevo archivo de metadata: /home/rafael/Dev/vasijas/data/10.cuadrados_divididos_en_train_val_y_test


## Vista previa al procesamiento

In [4]:
METADATA_ORIGEN = cargar_metadata(DATA_ORIGEN)
print("Muestra metadata origen:")
METADATA_ORIGEN.head()

Muestra metadata origen:


,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,Ubicacion
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero,Almedinilla
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero,Almedinilla
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero,Almedinilla
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero,Almedinilla
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero,Almedinilla


## Configuración del split

In [5]:
# Parametros del split
SEED = 42
CLASES_A_DIVIDIR = None  # None procesa todas las clases.

config_split = ConfiguracionSplitDataset(semilla=SEED)


def calcular_proporciones_split(
    configuracion: ConfiguracionSplitDataset,
) -> dict[str, float]:
    proporcion_validacion = configuracion.proporcion_temporal * (
        1 - configuracion.proporcion_test_desde_temporal
    )
    proporcion_test = (
        configuracion.proporcion_temporal * configuracion.proporcion_test_desde_temporal
    )

    return {
        "train": 1 - configuracion.proporcion_temporal,
        "val": proporcion_validacion,
        "test": proporcion_test,
    }


def describir_configuracion_split(configuracion: ConfiguracionSplitDataset) -> str:
    proporciones = calcular_proporciones_split(configuracion)
    return (
        "Split configurado -> "
        f"train: {proporciones['train']:.0%}, "
        f"val: {proporciones['val']:.0%}, "
        f"test: {proporciones['test']:.0%}, "
        f"semilla: {configuracion.semilla}"
    )


print(describir_configuracion_split(config_split))

Split configurado -> train: 70%, val: 15%, test: 15%, semilla: 42


## Construcción del split

La lógica reutilizable vive en `vasijas.split_dataset`. El notebook solo define parámetros, ejecuta la preparación del split y muestra los resultados.

Se usa `torchvision.datasets.ImageFolder` dentro del módulo para respetar el mismo orden de muestras y las mismas clases que consume el entrenamiento. `CLASES_A_DIVIDIR` permite limitar el split a un subconjunto de carpetas/clases. Luego se aplica `train_test_split` dos veces: primero `train` contra el conjunto temporal, y después `val` contra `test`.


In [6]:
resultado_split = preparar_metadata_split_perfiles_completos(
    data_origen=DATA_ORIGEN,
    metadata_origen=METADATA_ORIGEN,
    configuracion_split=config_split,
    clases_a_dividir=CLASES_A_DIVIDIR,
)

df_split = resultado_split.registros_split


def mostrar_resumen_split(resultado) -> None:
    conteos = resultado.conteos_split
    print("Clases procesadas:", resultado.clases_procesadas)
    print("Imagenes procesadas:", resultado.total_imagenes)
    print(
        f"Split -> train: {conteos['train']}, "
        f"val: {conteos['val']}, "
        f"test: {conteos['test']}"
    )


mostrar_resumen_split(resultado_split)

Clases procesadas: ['clase01', 'clase02', 'clase03', 'clase04', 'clase05', 'clase06', 'clase07', 'clase08', 'clase09', 'clase10', 'clase11', 'sin_clase']
Imagenes procesadas: 1242
Split -> train: 869, val: 186, test: 187


In [7]:
# Tabla de archivos con split
df_split.head()

,Identificador,archivo,ruta_relativa,clase_carpeta,split
0,al_13b,al_13b.png,clase01/al_13b.png,clase01,train
1,al_14b,al_14b.png,clase01/al_14b.png,clase01,train
2,ba_115_4,ba_115_4.png,clase01/ba_115_4.png,clase01,train
3,ba_172_3,ba_172_3.png,clase01/ba_172_3.png,clase01,train
4,ba_191_2,ba_191_2.png,clase01/ba_191_2.png,clase01,train


## Unión con metadata

La metadata se limita a las imágenes procesadas y se agregan las columnas `archivo`, `ruta_relativa`, `clase_carpeta` y `split`.


In [8]:
df_metadata_split = resultado_split.metadata


def advertir_imagenes_sin_metadata(ids_sin_metadata: list[str]) -> None:
    if not ids_sin_metadata:
        return

    print(
        "Advertencia: hay imagenes sin registro en metadata. "
        f"Cantidad: {len(ids_sin_metadata)}. "
        f"Primeros casos: {ids_sin_metadata[:10]}"
    )


advertir_imagenes_sin_metadata(resultado_split.ids_sin_metadata)
df_metadata_split.head()

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,Ubicacion,archivo,ruta_relativa,clase_carpeta,split
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero,Almedinilla,al_03a.png,clase11/al_03a.png,clase11,test
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero,Almedinilla,al_03b.png,clase11/al_03b.png,clase11,test
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero,Almedinilla,al_03c.png,clase11/al_03c.png,clase11,test
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero,Almedinilla,al_03d.png,clase11/al_03d.png,clase11,train
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero,Almedinilla,al_03e.png,clase11/al_03e.png,clase11,val


In [9]:
# Control de distribucion por split y clase
def construir_resumen_split(metadata_split: pd.DataFrame) -> pd.DataFrame:
    return pd.crosstab(
        metadata_split["clase_carpeta"],
        metadata_split["split"],
        margins=True,
    )


resumen_split = construir_resumen_split(df_metadata_split)
resumen_split

split,test,train,val,All
clase_carpeta,,,,
clase01,4,18,4,26
clase02,7,32,7,46
clase03,11,50,10,71
clase04,1,7,2,10
clase05,9,39,8,56
clase06,7,36,8,51
clase07,7,34,7,48
clase08,42,196,42,280
clase09,3,14,3,20


In [10]:
# Guardado de metadata reutilizable por otros notebooks
ruta_metadata_split = guardar_metadata(
    df=df_metadata_split,
    carpeta_destino=DATA_DESTINO,
)

print(f"Metadata con split guardada en: {ruta_metadata_split}")
print(f"Registros guardados: {len(df_metadata_split)}")

Metadata con split guardada en: /home/rafael/Dev/vasijas/data/10.cuadrados_divididos_en_train_val_y_test/metadata.csv
Registros guardados: 1242


## Uso desde otros notebooks

```python
import pandas as pd

from vasijas.config import load_config
from vasijas.paths import ProjectPaths

CONFIG = load_config()
PATHS = ProjectPaths.from_root(CONFIG.project_root)

metadata_split = pd.read_csv(PATHS.DATA_DIVIDIDO / "metadata.csv")
train_df = metadata_split[metadata_split["split"] == "train"]
val_df = metadata_split[metadata_split["split"] == "val"]
test_df = metadata_split[metadata_split["split"] == "test"]
```
